In [16]:
import pandas as pd
import json

def process_job_data(file_path):
    """
    Comprehensive data loading and cleaning pipeline for 1M+ records.
    Ensures all columns are retained and optimized for downstream insights.
    """
    # 1. Complete optimal data types to minimize memory across all features
    optimized_dtypes = {
        'employmentTypes': 'category',
        'metadata_isPostedOnBehalf': 'boolean',
        'metadata_jobPostId': 'string',
        'metadata_repostCount': 'Int16',
        'metadata_totalNumberJobApplication': 'Int32', 
        'metadata_totalNumberOfView': 'Int32',
        'minimumYearsExperience': 'Int8',
        'numberOfVacancies': 'Int16',
        'positionLevels': 'category',
        'postedCompany_name': 'string',
        'salary_maximum': 'float32',
        'salary_minimum': 'float32',
        'salary_type': 'category',
        'status_jobStatus': 'category',
        'title': 'string',
        'average_salary': 'float32'
    }
    
    # 2. Define date columns for automatic parsing (vital for time-series insights)
    date_cols = [
        'metadata_expiryDate', 
        'metadata_newPostingDate', 
        'metadata_originalPostingDate'
    ]

    print("Loading data...")
    # 3. Load the CSV file
    df = pd.read_csv(
        file_path,
        dtype=optimized_dtypes,
        parse_dates=date_cols,
        dayfirst=True,  # Ensures dates like 22/4/2023 are parsed correctly
        low_memory=False,
        nrows = 50000
    )
    
    # 4. Drop columns that are completely empty
    if 'occupationId' in df.columns:
        df = df.drop(columns=['occupationId'])
        print("Dropped empty 'occupationId' column.")

    # 5. Parse the JSON string in 'categories'
    print("Parsing JSON categories...")
    def extract_categories(json_str):
        if pd.isna(json_str):
            return []
        try:
            parsed = json.loads(json_str)
            return [item.get('category') for item in parsed if 'category' in item]
        except (json.JSONDecodeError, TypeError):
            return []

    # Apply extraction and drop the original raw JSON string column
    df['parsed_categories'] = df['categories'].apply(extract_categories)
    df = df.drop(columns=['categories'])
    
    print("Pipeline execution complete. DataFrame is ready for analysis.")
    return df

# Execute the function and store it in memory for all subsequent cells
# Replace 'jobs_dataset.csv' with your actual file
df_clean = process_job_data('SGJobData.csv')

# Verify the fully loaded and cleaned dataframe
df_clean.info()


Loading data...
Dropped empty 'occupationId' column.
Parsing JSON categories...
Pipeline execution complete. DataFrame is ready for analysis.
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   employmentTypes                     50000 non-null  category      
 1   metadata_expiryDate                 50000 non-null  str           
 2   metadata_isPostedOnBehalf           50000 non-null  boolean       
 3   metadata_jobPostId                  50000 non-null  string        
 4   metadata_newPostingDate             50000 non-null  str           
 5   metadata_originalPostingDate        50000 non-null  datetime64[us]
 6   metadata_repostCount                50000 non-null  Int16         
 7   metadata_totalNumberJobApplication  50000 non-null  Int32         
 8   metadata_totalNumberOfView         

C:\Users\nived\AppData\Local\Temp\ipykernel_14172\1880685850.py:38: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv(


Cell2: Mismatch cell analysis

In [17]:
import numpy as np

# 1. Explode the pre-parsed categories so each job can be counted under multiple sectors if necessary
df_exploded = df_clean.explode('parsed_categories')

# 2. Aggregate Market Signals by Sector
print("Calculating supply and demand metrics...")
sector_metrics = df_exploded.groupby('parsed_categories').agg(
    total_postings=('metadata_jobPostId', 'count'),
    total_vacancies=('numberOfVacancies', 'sum'),
    total_applications=('metadata_totalNumberJobApplication', 'sum'),
    avg_repost_count=('metadata_repostCount', 'mean'),
    avg_salary=('average_salary', 'mean'),
    avg_years_exp=('minimumYearsExperience', 'mean')
).reset_index()

# 3. Calculate Critical Efficiency Ratios
# Applications per Vacancy (Supply Density)
sector_metrics['apps_per_vacancy'] = (
    sector_metrics['total_applications'] / sector_metrics['total_vacancies']
).replace([np.inf, -np.inf], np.nan)

# 4. Identify Shortages (High Reposting, Low Supply)
# Sorting by highest repost count and lowest applications per vacancy
shortages = sector_metrics.sort_values(
    by=['avg_repost_count', 'apps_per_vacancy'], 
    ascending=[False, True]
).dropna(subset=['apps_per_vacancy'])

# 5. Display Results
columns_to_display = [
    'parsed_categories', 'total_vacancies', 'avg_repost_count', 
    'apps_per_vacancy', 'avg_salary'
]
print("\n--- Top 10 Sectors Experiencing Critical Skills Gaps ---")
print(shortages[columns_to_display].head(10).to_markdown(index=False))


Calculating supply and demand metrics...

--- Top 10 Sectors Experiencing Critical Skills Gaps ---
| parsed_categories                 |   total_vacancies |   avg_repost_count |   apps_per_vacancy |   avg_salary |
|:----------------------------------|------------------:|-------------------:|-------------------:|-------------:|
| Purchasing / Merchandising        |              1433 |           0.895238 |            7.27285 |      4271.83 |
| Social Services                   |               579 |           0.778426 |           16.9948  |      3569.6  |
| Manufacturing                     |              5837 |           0.742936 |            3.98115 |      4114.54 |
| Precision Engineering             |               533 |           0.653731 |            4.38274 |      4234.18 |
| Real Estate / Property Management |               904 |           0.650424 |           10.1681  |      4819.5  |
| Telecommunications                |              3005 |           0.631799 |            1.3034

In [18]:
print(df_clean.columns.tolist())

['employmentTypes', 'metadata_expiryDate', 'metadata_isPostedOnBehalf', 'metadata_jobPostId', 'metadata_newPostingDate', 'metadata_originalPostingDate', 'metadata_repostCount', 'metadata_totalNumberJobApplication', 'metadata_totalNumberOfView', 'minimumYearsExperience', 'numberOfVacancies', 'positionLevels', 'postedCompany_name', 'salary_maximum', 'salary_minimum', 'salary_type', 'status_id', 'status_jobStatus', 'title', 'average_salary', 'parsed_categories']


In [19]:
# Average salary for each job category
salary_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')['average_salary']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("\n===== Highest Paying Job Categories =====")
print(salary_by_sector.head(10).to_markdown(index=False))


===== Highest Paying Job Categories =====
| parsed_categories           |   average_salary |
|:----------------------------|-----------------:|
| Risk Management             |          7738.54 |
| Banking and Finance         |          7085.01 |
| Information Technology      |          6696.86 |
| Consulting                  |          6086.69 |
| Legal                       |          5940.12 |
| Insurance                   |          5500.82 |
| Professional Services       |          5195.62 |
| General Management          |          5019.9  |
| Engineering                 |          4926.19 |
| Sciences / Laboratory / R&D |          4851.59 |


In [20]:
# Most viewed job categories
views_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')['metadata_totalNumberOfView']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("\n===== Most Viewed Job Categories =====")
print(views_by_sector.head(10).to_markdown(index=False))


===== Most Viewed Job Categories =====
| parsed_categories                 |   metadata_totalNumberOfView |
|:----------------------------------|-----------------------------:|
| Social Services                   |                      338.586 |
| Public / Civil Service            |                      210.224 |
| Real Estate / Property Management |                      187.742 |
| Risk Management                   |                      185.814 |
| Banking and Finance               |                      183.408 |
| Advertising / Media               |                      176.59  |
| Travel / Tourism                  |                      175.845 |
| Wholesale Trade                   |                      167.848 |
| Design                            |                      165.554 |
| Professional Services             |                      159.172 |


In [21]:
experience_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')['minimumYearsExperience']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("\n===== Average Years of Experience Required =====")
print(experience_by_sector.head(10).to_markdown(index=False))


===== Average Years of Experience Required =====
| parsed_categories                 |   minimumYearsExperience |
|:----------------------------------|-------------------------:|
| Risk Management                   |                  4.45067 |
| Information Technology            |                  3.66509 |
| Banking and Finance               |                  3.62217 |
| Building and Construction         |                  3.61327 |
| Consulting                        |                  3.40034 |
| Real Estate / Property Management |                  3.34534 |
| General Management                |                  3.30946 |
| Engineering                       |                  3.22389 |
| Architecture / Interior Design    |                  3.06922 |
| Precision Engineering             |                  2.91343 |


In [22]:
#Field categories with the biggest talent shortages
high_demand = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')
    .agg(
        total_vacancies=('numberOfVacancies', 'sum'),
        total_applications=('metadata_totalNumberJobApplication', 'sum'),
        avg_reposts=('metadata_repostCount', 'mean')
    )
)

high_demand['applications_per_vacancy'] = (
    high_demand['total_applications'] /
    high_demand['total_vacancies']
)

high_demand = high_demand.sort_values(
    by=['total_vacancies', 'applications_per_vacancy'],
    ascending=[False, True]
)

print(high_demand.head(10))

                             total_vacancies  total_applications  avg_reposts  \
parsed_categories                                                               
Customer Service                       18780               49272     0.462343   
Information Technology                 15382               83202      0.37839   
Others                                 14800               44075     0.595758   
F&B                                    14627               17287     0.439557   
Engineering                            14202               57745     0.552073   
Sales / Retail                         12591               35384      0.38715   
Admin / Secretarial                    12510               73324      0.33611   
Healthcare / Pharmaceutical             9657               23537     0.506704   
Building and Construction               9446               37113     0.423218   
Human Resources                         8379               35219     0.296402   

                           

In [ ]:
#Most in demand sectors 
demand_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')
    .agg(
        total_jobs=('metadata_jobPostId','count'),
        total_vacancies=('numberOfVacancies','sum')
    )
    .sort_values(
        by='total_vacancies',
        ascending=False
    )
)

print("===== Most In-Demand Sectors =====")
print(demand_by_sector.head(10))

===== Most In-Demand Sectors =====
                             total_jobs  total_vacancies
parsed_categories                                       
Customer Service                   5258            18780
Information Technology             6969            15382
Others                             4762            14800
F&B                                3706            14627
Engineering                        7115            14202
Sales / Retail                     4856            12591
Admin / Secretarial                5403            12510
Healthcare / Pharmaceutical        2163             9657
Building and Construction          4083             9446
Human Resources                    2446             8379


In [26]:
print("===== Singapore Job Market Summary =====")

print(f"Total Job Postings: {df_clean['metadata_jobPostId'].nunique():,}")
print(f"Total Companies: {df_clean['postedCompany_name'].nunique():,}")
print(f"Total Vacancies: {df_clean['numberOfVacancies'].sum():,}")
print(f"Average Salary: ${df_clean['average_salary'].mean():,.2f}")
print(f"Average Applications per Job: {df_clean['metadata_totalNumberJobApplication'].mean():.2f}")
print(f"Average Views per Job: {df_clean['metadata_totalNumberOfView'].mean():.2f}")

===== Singapore Job Market Summary =====
Total Job Postings: 50,000
Total Companies: 11,185
Total Vacancies: 129,024
Average Salary: $4,582.15
Average Applications per Job: 10.87
Average Views per Job: 134.37
